# 🧠 CerebroSpheroid V4: Dual T4 GPU Training on Kaggle

### **Architecture & Enhancements in V4-Small:**
1. **Modern Llama 32,000 Vocabulary**: Byte-fallback BPE with atomic single-digit arithmetic (`0`-`9`) and zero OOV tokens (127M total parameters).
2. **Causal Temporal Sensory Ingestion**: Exponential recency decay + phase velocity momentum vectors replacing order-erasing centroid pooling.
3. **Riemannian SpheroidOptimizer**: Persistent power iteration spectral radius guard (<1.5 ms) and single-moment memory saving 50% RAM over AdamW.
4. **Curriculum Scheduled Sampling**: Smoothly transitions from teacher forcing ($p=1.0$) to internal deliberation ($p=0.0$) over 5,000 steps.
5. **Held-Out Validation & Direct Cloud Sync**: Continuously evaluates on OpenHermes-2.5 and GSM8K and mirrors checkpoints to `bioflo.in` without quota issues.

---
### ⚡ Kaggle Settings:
- **Accelerator**: Select **GPU T4 x2** (Dual GPU) or **GPU P100**
- **Internet**: Toggle **Internet ON**
- **Persistence**: Variables and Files

## Step 1: GPU Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")

## Step 2: Clone / Pull Latest Repository & Dependencies

In [ ]:
import os
if not os.path.exists("/kaggle/working/reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git /kaggle/working/reasoning_agent
%cd /kaggle/working/reasoning_agent
!git fetch origin main
!git reset --hard origin/main
!pip install -q pyarrow fastparquet transformers datasets sentencepiece accelerate optax "jax[cuda12]"


## Step 3: Run Acceptance Verification Tests

In [ ]:
!python3 -u training/test_v4_architecture.py
!python3 -u training/test_spheroid_optimizer.py

## Step 4: Run Dual-GPU Training (V4 Small, SpheroidOptimizer & Scheduled Sampling)

In [ ]:
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

# Launch High-Throughput JAX Engine with 3B Dense Knowledge Stream (40% Cosmo + 40% FineWeb-Edu + 20% Hermes)
!python3 training/train_v4_jax.py \n    --ckpt_dir /kaggle/working/checkpoints \n    --resume_target latest \n    --dataset_source dense_edu \n    --elastic_window \n    --total_window 1024 \n    --batch_size 16 \n    --grad_accum 4 \n    --lr 0.0005 \n    --deliberation_mode scheduled \n    --schedule_steps 20000 \n    --min_sampling_p 0.35 \n    --repetition_penalty 1.25 \n    --eval_interval 200 \n    --thalamic_anchor_weight 0.1 \n    --max_hours 8.5 \n    --cloud_save_hours 1.0 \n    --log_every 10 \n    --cloud_sync \n    --source_node kaggle_gpu


## Step 5: Package Checkpoints & Verify Bioflo Cloud Sync

In [ ]:
import os, glob, json
from src.cloud_sync import check_remote_manifest

!mkdir -p /kaggle/working/checkpoints_out
!cp /kaggle/working/reasoning_agent/checkpoints/*_best* /kaggle/working/checkpoints_out/ 2>/dev/null || true
!cp /kaggle/working/reasoning_agent/checkpoints/*_latest* /kaggle/working/checkpoints_out/ 2>/dev/null || true
!zip -r /kaggle/working/cerebro_spheroid_v4_small_kaggle.zip /kaggle/working/checkpoints_out
!ls -lh /kaggle/working/*.zip

print("\n🌐 Remote bioflo.in manifest:")
print(json.dumps(check_remote_manifest(), indent=2))
print("\n✅ Checkpoints successfully archived and mirrored!")

## Step 6: Test Model Inference (Conversational & Reasoning QA)
Tests the trained model using 1-token biological delimiters ( [20436] sensory prompt,  [1044] motor response).


In [ ]:
import torch
import glob
from src.corpus import build_tokenizer
from src.spheroid_v4 import build_spheroid_v4

ckpts = sorted(glob.glob("/kaggle/working/reasoning_agent/checkpoints/*_best.pt") or glob.glob("/kaggle/working/reasoning_agent/checkpoints/*.pt"))
if ckpts:
    ckpt_path = ckpts[-1]
    print(f"Loading checkpoint: {ckpt_path}")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tok = build_tokenizer("llama")
    model = build_spheroid_v4("v4_small", vocab_size=len(tok))
    model.load_compatible_checkpoint(ckpt_path, device=device)
    model.to(device=device, dtype=torch.float32)
    model.eval()

    questions = [
        "Hi",
        "Hello! Who are you?",
        "What is the capital of France?",
        "What is 2 + 3?",
        "Why is the sky blue?",
        "If you have 3 apples and eat 1, how many are left?",
    ]

    print("
🧬 Model Inference with Enclosed Biological Delimiters (<stim> / <act>):")
    print("=" * 65)
    for q in questions:
        ans = model.generate_text(
            q,
            tokenizer=tok,
            thinking_cycles=3,
            max_new_tokens=64,
            temperature=0.6,
            repetition_penalty=1.35,
            device=device,
            strip_bio_prefix=True,
        )
        print(f"User:      {q}")
        print(f"Spheroid:  {ans}")
        print("-" * 65)
else:
    print("No checkpoint found in checkpoints/")
